# PISA 2025 student questionnaire: safe first look

This notebook reads metadata, a tiny raw preview, and selected columns from the compact Parquet file. It does **not** load the full 2 GB SPSS file into memory.

In [2]:
from pathlib import Path
import pandas as pd
import pyreadstat
import pyarrow.parquet as pq

DATA = Path('../../data/raw/CY09_MS_STU_PUF.sav')
CORE = Path('../data/processed/student_core.parquet')
DATA, CORE

(PosixPath('../data/raw/CY09_MS_STU_PUF.sav'),
 PosixPath('../data/processed/student_core.parquet'))

In [3]:
_, meta = pyreadstat.read_sav(DATA, metadataonly=True)
print(f'Rows: {meta.number_rows:,}')
print(f'Columns: {meta.number_columns:,}')

Rows: 755,721
Columns: 885


In [4]:
schema = pd.DataFrame({
    'variable': meta.column_names,
    'label': [meta.column_names_to_labels.get(c) for c in meta.column_names],
    'type': [meta.readstat_variable_types.get(c) for c in meta.column_names],
    'measure': [meta.variable_measure.get(c) for c in meta.column_names],
})
schema.head(30)

,variable,label,type,measure
0,CNT,Country code 3-character,string,nominal
1,CNTRYID,3-digit country code,string,nominal
2,CNTSCHID,Intl. School ID,string,nominal
3,CNTSTUID,Intl. Student ID,string,nominal
4,CYC,PISA Assessment Cycle (2 digits + 2 character ...,string,nominal
5,NatCen,National Centre 6-digit Code,string,nominal
6,STRATUM,Stratum ID 5-character (cnt + original stratum...,string,nominal
7,SUBNATIO,Adjudicated sub-region code 7-digit code (3-di...,string,nominal
8,REGION,REGION,string,nominal
9,OECD,OECD country,double,scale


In [5]:
# Search variable names and questionnaire labels without reading student records.
schema[schema['label'].fillna('').str.contains('gender|belong|anxiety', case=False, regex=True)].head(30)

,variable,label,type,measure
24,ST004D01T,Students' gender from sampling data (Female/Male),double,scale
25,MALE,Students' gender from sampling data (Male/Other),double,scale
120,ST034Q03TA,Thinking about your school: to what extent do ...,double,scale
387,IC181Q02JA,How upset were you the last time the following...,double,scale
586,PA204Q04DA,To what extent do you agree or disagree with t...,double,scale
607,BELONG,Sense of belonging (WLE),double,scale


In [6]:
preview_columns = schema['variable'].head(12).tolist()
preview, preview_meta = pyreadstat.read_sav(
    DATA, usecols=preview_columns, row_limit=20, apply_value_formats=False
)
preview

,CNT,CNTRYID,CNTSCHID,CNTSTUID,CYC,NatCen,STRATUM,SUBNATIO,REGION,OECD,ADMINMODE,LANGTEST_COG
0,ALB,008,00800001,00800001,25MS,000800,ALB01,0080000,00800,0.0,1.0,sqi
1,ALB,008,00800001,00800002,25MS,000800,ALB01,0080000,00800,0.0,1.0,sqi
2,ALB,008,00800001,00800003,25MS,000800,ALB01,0080000,00800,0.0,1.0,sqi
3,ALB,008,00800001,00800004,25MS,000800,ALB01,0080000,00800,0.0,1.0,sqi
4,ALB,008,00800001,00800005,25MS,000800,ALB01,0080000,00800,0.0,1.0,sqi
5,ALB,008,00800001,00800006,25MS,000800,ALB01,0080000,00800,0.0,1.0,sqi
6,ALB,008,00800001,00800007,25MS,000800,ALB01,0080000,00800,0.0,1.0,sqi
7,ALB,008,00800001,00800008,25MS,000800,ALB01,0080000,00800,0.0,1.0,sqi
8,ALB,008,00800001,00800009,25MS,000800,ALB01,0080000,00800,0.0,1.0,sqi
9,ALB,008,00800001,00800010,25MS,000800,ALB01,0080000,00800,0.0,1.0,sqi


## Fast preview of the compact analysis file

In [7]:
core_file = pq.ParquetFile(CORE)
print(f'Rows: {core_file.metadata.num_rows:,}')
print(f'Columns: {core_file.metadata.num_columns:,}')

display_columns = [
    'CNT', 'CNTSCHID', 'CNTSTUID', 'ST004D01T', 'AGE', 'ESCS',
    'W_FSTUWT', 'PV1SCIE', 'PV1MATH', 'PV1READ'
]
first_batch = next(core_file.iter_batches(batch_size=20, columns=display_columns))
first_batch.to_pandas()

Rows: 755,721
Columns: 132


,CNT,CNTSCHID,CNTSTUID,ST004D01T,AGE,ESCS,W_FSTUWT,PV1SCIE,PV1MATH,PV1READ
0,ALB,00800001,00800001,2.0,15.83,-1.100674,2.579892,289.678433,328.701396,283.163374
1,ALB,00800001,00800002,1.0,15.92,-1.930787,2.681028,416.224453,384.288713,410.218507
2,ALB,00800001,00800003,1.0,15.83,-1.078518,2.681028,488.596531,438.355683,423.163736
3,ALB,00800001,00800004,1.0,16.25,-0.018204,2.681028,326.012761,344.159139,421.192543
4,ALB,00800001,00800005,1.0,15.50,-1.071648,2.681028,348.666297,376.320312,331.749128
5,ALB,00800001,00800006,2.0,16.17,-0.085581,2.579892,238.070007,273.542173,190.970365
6,ALB,00800001,00800007,2.0,16.00,NaN,2.579892,335.034486,358.651019,317.468107
7,ALB,00800001,00800008,2.0,15.83,-1.236167,2.579892,341.961987,285.081969,339.799526
8,ALB,00800001,00800009,2.0,15.33,-2.176807,2.579892,315.433750,315.449494,246.257571
9,ALB,00800001,00800010,2.0,15.58,-0.735643,2.579892,470.050650,456.523875,450.991974


In [8]:
# Reading one selected column from Parquet is quick.
countries = pd.read_parquet(CORE, columns=['CNT'])
countries['CNT'].value_counts().rename_axis('country').to_frame('students').head(20)

,students
country,
ARE,34639
BRA,30140
ESP,29966
CAN,24601
KAZ,22680
GBR,15546
AUS,14658
IDN,14168
QCI,12385
